# Multiple Correspondence Analysis with Homals

Python equivalent of R Gifi vignette: *Multiple Correspondence Analysis*

**Homals** (Homogeneity Analysis) finds low-dimensional representations of
nominal categorical data by simultaneously scaling object scores and category
quantifications so that objects in the same category are close together.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pygifi import Homals, get_dataset
from pygifi.visualization.plot import plot

## 1 · ABC Customer Satisfaction Data

The ABC dataset contains satisfaction ratings (nominal, 3–5 categories) for 3 service aspects from 2176 customers.

In [ ]:
abc = get_dataset('abc')
print(abc.shape)
print(abc.head())
print(abc.dtypes)

### 1.1 · Fit Homals

In [ ]:
h_abc = Homals(ndim=2, normobj_z=True, eps=1e-8, itmax=1000).fit(abc)
print(h_abc)
print('\nEigenvalues:', np.round(h_abc.eigenvalues_[:4], 4))

### 1.2 · Category Quantifications Plot (loadplot)

Category points are plotted in the 2-dimensional solution space. Points close together represent categories that co-occur frequently.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
plot(h_abc, plot_type='loadplot', ax=ax, title='ABC — Category Quantifications')
plt.tight_layout()
plt.show()

### 1.3 · Object Scores Plot (objplot)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
plot(h_abc, plot_type='objplot', ax=ax, title='ABC — Object Scores')
plt.tight_layout()
plt.show()

### 1.4 · D-Measures (discrimination measures)

D-measures quantify how much each variable discriminates between objects in each dimension.

In [ ]:
h_abc.summary()

In [ ]:
# dmeasures is a list of arrays (one per variable)
dmeasures = h_abc.result_['dmeasures']
var_names = list(abc.columns)
dm_arr = np.array([d.diagonal() for d in dmeasures])  # (n_vars, ndim)

x = np.arange(len(var_names))
width = 0.35
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - width/2, dm_arr[:, 0], width, label='Dim 1', color='steelblue')
ax.bar(x + width/2, dm_arr[:, 1], width, label='Dim 2', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(var_names)
ax.set_ylabel('Discrimination Measure')
ax.set_title('ABC -- Discrimination Measures')
ax.legend()
plt.tight_layout()
plt.show()

---

## 2 · GALO School Data

The GALO dataset (`n=1290`) contains school data: gender, IQ score, and school advice. The `advice` variable has 7 ordered categories.

In [ ]:
galo = get_dataset('galo')
print(galo.shape)
print(galo.head())
print('\nAdvice categories:', sorted(galo['advice'].unique()))

In [ ]:
h_galo = Homals(ndim=2, normobj_z=True, eps=1e-8, itmax=1000).fit(galo)
print(h_galo)
print('\nStress:', round(h_galo.result_['f'], 5))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot(h_galo, plot_type='loadplot', ax=axes[0], title='GALO — Category Points')
plot(h_galo, plot_type='objplot', ax=axes[1], title='GALO — Object Scores')
plt.tight_layout()
plt.show()

### 2.1 · Projection Plot (prjplot)

Category centroids projected into object score space — one panel per variable.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
plot(h_galo, plot_type='prjplot', ax=ax, title='GALO — Projection Plot')
plt.tight_layout()
plt.show()

### 2.2 · Object scores coloured by gender

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
plot(h_galo, plot_type='objplot', ax=ax, group=galo['gender'],
     title='GALO — Object Scores by Gender')
plt.tight_layout()
plt.show()

---

## 3 · Ties Handling

Gifi supports three tie-handling modes for isotone regression:
- `'s'` — secondary (pool groups, PAVA on group means; **default**)
- `'p'` — primary (PAVA over all individual observations)
- `'t'` — tertiary (translate group means after PAVA)

In [ ]:
from pygifi import Princals

stresses = {}
for ties in ['s', 'p', 't']:
    m = Princals(ndim=2, ordinal=True, ties=ties, eps=1e-8, itmax=500).fit(galo)
    stresses[ties] = round(m.result_['f'], 5)

print('Stress by tie mode:')
for k, v in stresses.items():
    print(f'  ties={k!r}: {v}')

---

## 4 · References

- Mair, P., De Leeuw, J. (2010). Homogeneity analysis in R: The package homals. *Journal of Statistical Software*, 31(4).
- De Leeuw, J. (1984). The Gifi system of nonlinear multivariate analysis. In *Data Analysis and Informatics III* (pp. 415–424).